In [1]:
import os
import openreview
from tqdm import tqdm
from datetime import datetime
import pandas as pd
from time import sleep

# Test OpenReview API

In [2]:
client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

In [3]:
name = "Lovkush Agarwal"
profiles = client.search_profiles(term=name)

In [4]:
print(len(profiles))
print(profiles[0])

1
{'active': True,
 'content': {'emails': ['****@gmail.com'],
             'emailsConfirmed': ['****@gmail.com'],
             'gender': 'Male',
             'gscholar': 'https://scholar.google.com/citations?view_op=list_works&hl=en&user=nm-XFVoAAAAJ',
             'history': [{'end': None,
                          'institution': {'country': 'GB',
                                          'domain': 'lovkush.com',
                                          'name': 'Independent'},
                          'position': 'Researcher',
                          'start': 2024},
                         {'end': 2020,
                          'institution': {'country': 'GB',
                                          'domain': 'le.ac.uk',
                                          'name': 'University of Leicester'},
                          'position': 'Instructor',
                          'start': 2017},
                         {'end': 2016,
                          'institution': {'countr

In [5]:
author_id = '~Lovkush_Agarwal1'
papers = client.get_all_notes(
    content={'authorids': author_id},
    details='replies'
)

In [6]:
print(papers[0])

{'cdate': 1724838331162,
 'content': {'TLDR': {'value': 'We study how language models might encode '
                               'paragraphs, and find newline tokens '
                               'activations do this to some extent.'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'pochinkov2024extracting,\n'
                                  'title={Extracting Paragraphs from {LLM} '
                                  'Token Activations},\n'
                                  'author={Nicky Pochinkov and Angelo Benoit '
                                  'and Lovkush Agarwal and Zainab Ali Majid '
                                  'and Lucile Ter-Minassian},\n'
                                  'booktitle={🍃 MINT: Foundation Model '
                                  'Interventions},\n'
                                  'year={2024},\n'
                                  'url={https://openreview.net/forum?id=4b675AHcqq}\n'
                   

In [7]:
results = {}
for paper in papers:
    paper_info = {
        'title': paper.content['title']['value'],
        'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
        # 'paperid': paper.id,
        'venueid': paper.content['venueid']['value'],
        'authorids': paper.content['authorids']['value'],
    }
    # Check for decision in replies
    decision = None
    for reply in paper.details['replies']:
        try:
            decision = reply['content']['decision']['value']
            break
        except:
            pass
    paper_info['decision'] = decision
    results[paper.id] = paper_info

for result in results:
    print(result)
    print(results[result])
    print()

4b675AHcqq
{'title': 'Extracting Paragraphs from LLM Token Activations', 'date_creation': '2024-08-28', 'venueid': 'NeurIPS.cc/2024/Workshop/MINT', 'authorids': ['~Nicky_Pochinkov1', '~Angelo_Benoit1', '~Lovkush_Agarwal1', '~Zainab_Ali_Majid1', '~Lucile_Ter-Minassian1'], 'decision': 'Accept'}



In [8]:
# based on later investigations, create new hacky function to get decision

def get_decision(paper: openreview.api.client.Note) -> tuple[str|None, str|None]:
    decision = None

    # first try to get decision from replies
    for reply in paper.details['replies']:
        try:
            decision = reply['content']['decision']['value']
            return decision, "From decision field in replies"
        except:
            pass
    
    # if that fails, try to get decision from replies but with recommendation field
    for reply in paper.details['replies']:
        try:
            recommendation = reply['content']['recommendation']['value']
            return recommendation, "From recommendation field in replies"
        except:
            pass
    
    # if that fails, try to see if 'Poster' or 'Oral' or 'Spotlight' is at the end of venue
    if 'venue' in paper.content:
        if paper.content['venue']['value'].lower().endswith('poster'):
            return 'Accept Poster', "From venue field"
        elif paper.content['venue']['value'].lower().endswith('oral'):
            return 'Accept Oral', "From venue field"
        elif paper.content['venue']['value'].lower().endswith('spotlight'):
            return 'Accept Spotlight', "From venue field"
    
    return None, "No decision found"
    

# Get names from Airtable

In [11]:
import os
import dotenv
from pyairtable import Api

dotenv.load_dotenv()
api = Api(os.environ['AIRTABLE_API_KEY'])

base_id = "appZq2f1sM0tW9kH7"
table_id = "tblX2K7X7sFF8Q8Be"
table = api.table(base_id, table_id)
table_data = table.all()


In [12]:
names = []
for row in table_data:
    name = row['fields']['Scholar Name']
    name = name.strip().lower() # Remove leading and trailing whitespace, make lower case
    if name == "":
        continue
    if name not in names:
        names.append(name)

# for name in names:
#     print(name)


# Get author ids from all names

In [13]:
# scholar info dict
# keys are names, value is dictionary with two keys: n_profiles and author_id if n_profiles is 1, otherwise None

client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

scholar_info = {}

for name in tqdm(names):
    profiles = client.search_profiles(term=name)
    n_profiles = len(profiles)
    if n_profiles == 1:
        author_id = profiles[0].id
    else:
        author_id = None 
    scholar_info[name] = {
        'n_profiles': n_profiles,
        'author_id': author_id
    }

  0%|          | 0/306 [00:00<?, ?it/s]

 65%|██████▌   | 200/306 [00:41<00:16,  6.59it/s]

Retrying request: GET /profiles/search?term=bilal+chughtai&es=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 201 requests, surpassing the limit of 200 requests. Please try again in 19 seconds (2025-10-31-6000555)', 'status': 429, 'details': {'limit': 200, 'remaining': 0, 'resetTime': '2025-10-31T15:10:37.672Z', 'used': 201, 'current': 201, 'reqId': '2025-10-31-6000555'}}, error: None


100%|██████████| 306/306 [01:23<00:00,  3.67it/s]


In [14]:
# count how many scholars have 1 profile
n_scholars_with_1_profile = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 1)
print(f"Number of scholars with 1 profile: {n_scholars_with_1_profile}")

# count how many scholars have 0 profiles
n_scholars_with_0_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 0)
print(f"Number of scholars with 0 profiles: {n_scholars_with_0_profiles}")

# count how many scholars have 2 or more profiles
n_scholars_with_2_or_more_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] >= 2)
print(f"Number of scholars with 2 or more profiles: {n_scholars_with_2_or_more_profiles}")

assert n_scholars_with_1_profile + n_scholars_with_0_profiles + n_scholars_with_2_or_more_profiles == len(names)

Number of scholars with 1 profile: 184
Number of scholars with 0 profiles: 75
Number of scholars with 2 or more profiles: 47


In [15]:
# for name, scholar in list(scholar_info.items())[:30]:
#     print(f"{name}, {scholar['n_profiles']}, {scholar['author_id']}")


In [16]:
# get list of author_ids where they are not None
author_ids = [scholar['author_id'] for scholar in scholar_info.values() if scholar['author_id'] is not None]
print(len(author_ids))



184


# get paper stats for all author_ids

In [17]:
all_papers = {}

for author_id in tqdm(author_ids):
    # wait to prevent rate limiting. seems to be limit of 60 requests per minute
    sleep(1)
    
    papers = client.get_all_notes(
        content={'authorids': author_id},
        details='replies'
    )

    for paper in papers:
        if paper.id in all_papers:
            continue
        paper_info = {
            'title': paper.content['title']['value'],
            'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
            # 'paperid': paper.id,
            'venueid': paper.content['venueid']['value'],
            'authorids': paper.content['authorids']['value'],
        }
        decision, decision_reasoning = get_decision(paper)
        paper_info['decision'] = decision
        paper_info['decision_reasoning'] = decision_reasoning
        all_papers[paper.id] = paper_info

 19%|█▉        | 35/184 [00:47<02:56,  1.19s/it]

Retrying request: GET /notes?content.authorids=~Thomas_Bush1&limit=1000&details=replies&after=DzGe40glxs&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 13 seconds (2025-10-31-6010265)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-31T15:12:01.868Z', 'used': 61, 'current': 61, 'reqId': '2025-10-31-6010265'}}, error: None


 36%|███▋      | 67/184 [01:45<02:46,  1.42s/it]

Retrying request: GET /notes?content.authorids=~Jai_Dhyani1&limit=1000&details=replies&after=3rB0bVU6z6&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 15 seconds (2025-10-31-6016795)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-31T15:13:02.437Z', 'used': 61, 'current': 61, 'reqId': '2025-10-31-6016795'}}, error: None


 55%|█████▍    | 101/184 [02:48<01:51,  1.34s/it]

...

 75%|███████▌  | 138/184 [03:50<00:56,  1.23s/it]

Retrying request: GET /notes?content.authorids=~Joe_Needham1&limit=1000&details=replies&after=NvhyJOWM8k&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 11 seconds (2025-10-31-6030860)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-31T15:15:03.195Z', 'used': 61, 'current': 61, 'reqId': '2025-10-31-6030860'}}, error: None


 93%|█████████▎| 172/184 [04:49<00:18,  1.53s/it]

Retrying request: GET /notes?content.authorids=~Carson_Ezell1&limit=1000&details=replies&sort=id&count=true, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 13 seconds (2025-10-31-6037610)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-31T15:16:03.897Z', 'used': 61, 'current': 61, 'reqId': '2025-10-31-6037610'}}, error: None


100%|██████████| 184/184 [05:22<00:00,  1.75s/it]


In [100]:
# convert dict to dataframe
df = pd.DataFrame(all_papers).T
df.reset_index(inplace=True)
df.rename(columns={'index': 'paperid'}, inplace=True)
df.head()

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral),From decision field in replies
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster),From decision field in replies
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster),From decision field in replies
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster),From decision field in replies
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster),From decision field in replies


In [101]:
# if venueid is `ICLR.cc/2025/Conference/Withdrawn_Submission` or
# `ICLR.cc/2025/Conference/Rejected_Submission` then set decision to 'reject' and 'decision_reasonsing' to 'withdrawn'

import re

withdrawn_patterns = [
    r'ICLR\.cc/\d{4}/Conference/Withdrawn_Submission',
    r'ICLR\.cc/\d{4}/Conference/Rejected_Submission',
    r'ICLR\.cc/\d{4}/Conference/Desk_Rejected_Submission'
]

# Boolean mask for any withdrawn/rejected/desk rejected submission, any year
mask = pd.Series(False, index=df.index)
for pattern in withdrawn_patterns:
    mask = mask | df['venueid'].str.match(pattern)

# Refine mask to only include rows where decision is NaN
mask = mask & df['decision'].isna()

# Set decision and reasoning for all matching rows where decision is NaN
df.loc[mask, 'decision'] = 'reject'
df.loc[mask, 'decision_reasoning'] = 'ICLR venueid contains withdrawn or rejected'

df.loc[mask, 'decision_simple'] = 'reject'


In [102]:
df['paper_url'] = df['paperid'].apply(lambda x: f"https://openreview.net/forum?id={x}")
df['decision_simple'] = df['decision'].apply(lambda x: x.split(' ')[0] if x else None).apply(lambda x: x.split('-')[0] if x else None).fillna("UNKNOWN").str.lower()
df.head()

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral),From decision field in replies,accept,https://openreview.net/forum?id=qzsDKwGJyB
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster),From decision field in replies,accept,https://openreview.net/forum?id=qrU3yNfX0d
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster),From decision field in replies,accept,https://openreview.net/forum?id=pZakRK1QHU
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster),From decision field in replies,accept,https://openreview.net/forum?id=bKawydfGhb
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster),From decision field in replies,accept,https://openreview.net/forum?id=SCEdoGghcw


In [103]:
print(len(df))

576


In [104]:
df['decision_simple'].value_counts(dropna=False)

decision_simple
accept     359
unknown    148
reject      69
Name: count, dtype: int64

In [105]:
df['decision_reasoning'].value_counts(dropna=False)

decision_reasoning
From decision field in replies                 247
From venue field                               150
No decision found                              148
From recommendation field in replies            16
ICLR venueid contains withdrawn or rejected     15
Name: count, dtype: int64

In [106]:
df.to_csv('all_papers.csv', index=False)

# basic analysis by conference

In [107]:
df = pd.read_csv('all_papers.csv')
df.head()

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"['~Adam_Karvonen1', '~Benjamin_Wright2', '~Can...",Accept (Oral),From decision field in replies,accept,https://openreview.net/forum?id=qzsDKwGJyB
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"['~Adam_Karvonen1', '~Can_Rager1', '~Johnny_Li...",Accept (poster),From decision field in replies,accept,https://openreview.net/forum?id=qrU3yNfX0d
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"['~Michael_Ivanitskiy1', '~Alexander_F_Spies1'...",Accept (Poster),From decision field in replies,accept,https://openreview.net/forum?id=pZakRK1QHU
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"['~Jett_Janiak1', '~Can_Rager1', '~James_Dao1'...",Accept (Poster),From decision field in replies,accept,https://openreview.net/forum?id=bKawydfGhb
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"['~Adam_Karvonen1', '~Benjamin_Wright2', '~Can...",Accept (poster),From decision field in replies,accept,https://openreview.net/forum?id=SCEdoGghcw


In [108]:
venue_counts = df['venueid'].value_counts(dropna=False)
venue_counts.head(30)


venueid
NeurIPS.cc/2025/Workshop/MechInterp              38
dblp.org/journals/CORR/2024                      34
ICLR.cc/2025/Conference/Rejected_Submission      33
ICML.cc/2025/Conference                          28
ICLR.cc/2025/Conference                          26
NeurIPS.cc/2025/Conference                       24
ICML.cc/2024/Workshop/MI                         21
NeurIPS.cc/2024/Conference                       17
TMLR                                             16
ICLR.cc/2024/Conference/Rejected_Submission      13
NeurIPS.cc/2024/Workshop/SoLaR                   12
dblp.org/journals/CORR/2025                      12
ICML.cc/2025/Workshop/R2-FM                      10
OpenReview.net/Archive                           10
ICLR.cc/2025/Conference/Withdrawn_Submission     10
dblp.org/journals/CORR/2023                       9
NeurIPS.cc/2024/Workshop/SafeGenAi                8
ICML.cc/2024/Conference                           8
EMNLP/2023/Conference                             7
ICLR

In [109]:
conferences = [
    'NeurIPS',
    'NeurIPS.cc/2025/Conference',
    'NeurIPS.cc/2024/Conference',
    'NeurIPS.cc/2023/Conference',
    'ICML',
    'ICML.cc/2025/Conference',
    'ICML.cc/2024/Conference',
    'ICML.cc/2023/Conference',
    'ICLR',
    'ICLR.cc/2025/Conference',
    'ICLR.cc/2024/Conference',
    'ICLR.cc/2023/Conference',
    'NeurIPS.cc/2025/Workshop/MechInterp',
    'dblp.org/journals/CORR',
    'ICML.cc/2024/Workshop/MI',
    'TMLR',
    'NeurIPS.cc/2025/Workshop/LLM_Evaluation',
    'NeurIPS.cc/2024/Workshop/SoLaR',
    'ICML.cc/2025/Workshop',
]

for conference in conferences:
    print(f"Venue id contains the string: {conference}")
    conference_decisions = df.loc[df['venueid'].str.contains(conference), 'decision_simple'].value_counts(dropna=False)

    # loop through and print info
    for decision, count in conference_decisions.items():
        print(f"{decision}: {count}")
    print()

Venue id contains the string: NeurIPS
accept: 159
unknown: 12
reject: 7

Venue id contains the string: NeurIPS.cc/2025/Conference
accept: 24
reject: 2

Venue id contains the string: NeurIPS.cc/2024/Conference
accept: 17
reject: 2

Venue id contains the string: NeurIPS.cc/2023/Conference
accept: 6

Venue id contains the string: ICML
accept: 99
unknown: 8

Venue id contains the string: ICML.cc/2025/Conference
accept: 28

Venue id contains the string: ICML.cc/2024/Conference
accept: 8

Venue id contains the string: ICML.cc/2023/Conference
accept: 3

Venue id contains the string: ICLR
reject: 62
accept: 51
unknown: 19

Venue id contains the string: ICLR.cc/2025/Conference
reject: 44
accept: 26

Venue id contains the string: ICLR.cc/2024/Conference
reject: 17
accept: 7

Venue id contains the string: ICLR.cc/2023/Conference

Venue id contains the string: NeurIPS.cc/2025/Workshop/MechInterp
accept: 38

Venue id contains the string: dblp.org/journals/CORR
unknown: 60

Venue id contains the str

In [110]:
df[(df['venueid'].str.contains('ICLR.cc/2025/Conference')) & (df['decision_simple']!='accept')]

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
13,V892sBHUbN,Rapid Response: Mitigating LLM Jailbreaks With...,2024-09-24,ICLR.cc/2025/Conference/Rejected_Submission,"['~Alwin_Peng1', '~Julian_Michael1', '~Henry_S...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=V892sBHUbN
24,0ULf242ApE,From Context to Concept: Concept Encoding in I...,2024-09-26,ICLR.cc/2025/Conference/Rejected_Submission,"['~Jinyeop_Song1', '~Seungwook_Han1', '~Jeff_G...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=0ULf242ApE
25,uDjuCpQH5N,Do Unlearning Methods Remove Information from ...,2024-09-24,ICLR.cc/2025/Conference/Rejected_Submission,"['~Aghyad_Deeb1', '~Fabien_Roger1']",Reject,From decision field in replies,reject,https://openreview.net/forum?id=uDjuCpQH5N
36,oycEeFXX74,Shell Games: Control Protocols for Adversarial...,2024-09-28,ICLR.cc/2025/Conference/Withdrawn_Submission,"['~Aryan_Bhatt1', '~Cody_Rushing1', '~Adam_Kau...",reject,ICLR venueid contains withdrawn or rejected,reject,https://openreview.net/forum?id=oycEeFXX74
50,lcF4BkhPBv,Algorithm for Concept Extrapolation: Diverse G...,2024-09-27,ICLR.cc/2025/Conference/Rejected_Submission,"['~Oliver_Daniels1', '~Stuart_Armstrong1', '~A...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=lcF4BkhPBv
67,wI5uHZLeCZ,Latent Adversarial Training Improves Robustnes...,2024-09-26,ICLR.cc/2025/Conference/Rejected_Submission,"['~Abhay_Sheshadri1', '~Aidan_Ewart1', '~Phill...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=wI5uHZLeCZ
68,vsU2veUpiR,Mechanistic Unlearning: Robust Knowledge Unlea...,2024-09-27,ICLR.cc/2025/Conference/Rejected_Submission,"['~Phillip_Huang_Guo1', '~Aaquib_Syed1', '~Abh...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=vsU2veUpiR
85,HgSIfXTpBE,Mixed-curvature decision trees and random forests,2024-09-27,ICLR.cc/2025/Conference/Rejected_Submission,"['~Philippe_Chlenski1', '~Quentin_Chu1', '~Rai...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=HgSIfXTpBE
115,4JBEpP6eRS,ZIP-FIT: Embedding-Free Data Selection via Com...,2024-09-27,ICLR.cc/2025/Conference/Rejected_Submission,"['~Elyas_Obbad1', '~Iddah_Mlauzi1', '~Brando_M...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=4JBEpP6eRS
132,dlUjNdybnq,Mitigating the Influence of Distractor Tasks i...,2024-09-27,ICLR.cc/2025/Conference/Rejected_Submission,"['~Raymond_Douglas1', '~Andis_Draguns1', '~Tom...",Reject,From decision field in replies,reject,https://openreview.net/forum?id=dlUjNdybnq


In [111]:
df[(df['venueid'].str.contains('dblp.org/journals/CORR/2024')) & (df['decision'].isna())].head(5)

# doing some searching seems to show that dblp is just some subset of arxiv?? and openreview for some reason indexes this...? does not actually seem to be any kind of review or conference.

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
19,8Pf40Qhbq5,Separating Tongue from Thought: Activation Pat...,2024-01-01,dblp.org/journals/CORR/2024,"['~Clément_Dumas1', '~Chris_Wendler1', 'https:...",NaN,No decision found,unknown,https://openreview.net/forum?id=8Pf40Qhbq5
29,a0Dw4niLjh,Obfuscated Activations Bypass LLM Latent-Space...,2024-01-01,dblp.org/journals/CORR/2024,"['~Luke_Bailey1', '~Alex_Serrano1', 'https://d...",NaN,No decision found,unknown,https://openreview.net/forum?id=a0Dw4niLjh
30,JJhPn1AgUi,xCOMET-lite: Bridging the Gap Between Efficien...,2024-01-01,dblp.org/journals/CORR/2024,['https://dblp.org/search/pid/api?q=author:Dan...,NaN,No decision found,unknown,https://openreview.net/forum?id=JJhPn1AgUi
32,9u9Zws3wGP,ViSTa Dataset: Do vision-language models under...,2024-01-01,dblp.org/journals/CORR/2024,['https://dblp.org/search/pid/api?q=author:Evz...,NaN,No decision found,unknown,https://openreview.net/forum?id=9u9Zws3wGP
131,iktWqWaheH,Limitations of Agents Simulated by Predictive ...,2024-01-01,dblp.org/journals/CORR/2024,['https://dblp.org/search/pid/api?q=author:Ray...,NaN,No decision found,unknown,https://openreview.net/forum?id=iktWqWaheH


In [112]:
df[(df['venueid'].str.contains('TMLR')) & (df['decision'].isna())].head(5)

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
442,KQOHBsXe6b,Beyond the Imitation Game: Quantifying and ext...,2023-01-01,dblp.org/journals/TMLR/2023,['https://dblp.org/search/pid/api?q=author:Aar...,NaN,No decision found,unknown,https://openreview.net/forum?id=KQOHBsXe6b
496,lPOQI7Kcb3,Inverse Scaling: When Bigger Isn't Better,2023-01-01,dblp.org/journals/TMLR/2023,"['~Ian_R._McKenzie1', '~Alexander_Lyzhov1', 'h...",NaN,No decision found,unknown,https://openreview.net/forum?id=lPOQI7Kcb3


In [113]:
df[(df['venueid'].str.contains('ICML.cc/2025/Workshop')) & (df['decision'].isna())].head(5)

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,decision_simple,paper_url
182,aMaSHy8IgK,Probing the Limits of Mathematical World Model...,2025-05-20,ICML.cc/2025/Workshop/World_Models,"['~Henry_Kvinge1', '~Elizabeth_Coda1', '~Eric_...",NaN,No decision found,unknown,https://openreview.net/forum?id=aMaSHy8IgK
183,VbqocgftJF,Permutations as a testbed for studying the eff...,2025-05-26,ICML.cc/2025/Workshop/MOSS,"['~Sarah_McGuire_Scullen1', '~Davis_Brown1', '...",NaN,No decision found,unknown,https://openreview.net/forum?id=VbqocgftJF
419,77w4a4oxpq,Evaluating Forecasting is More Difficult than ...,2025-05-22,ICML.cc/2025/Workshop/World_Models,"['~Daniel_Paleka1', '~Shashwat_Goel1', '~Jonas...",NaN,No decision found,unknown,https://openreview.net/forum?id=77w4a4oxpq


In [114]:
# can maybe use the invitations field to get information??
# 'invitations': ['ICML.cc/2025/Workshop/World_Models/-/Submission',
                #  'ICML.cc/2025/Workshop/World_Models/-/Post_Submission',
                #  'ICML.cc/2025/Workshop/World_Models/-/Edit',
                #  'ICML.cc/2025/Workshop/World_Models/Submission7/-/Camera-Ready'],
paper_id = "aMaSHy8IgK"
paper = client.get_all_notes(id=paper_id, details='replies')
print(paper[0])


{'cdate': 1747698466524,
 'content': {'TLDR': {'value': 'We investigate whether the mathematical world '
                               'models of LLMs align with structures and '
                               'properties from mathematics broadly'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'kvinge2025probing,\n'
                                  'title={Probing the Limits of Mathematical '
                                  'World Models in {LLM}s},\n'
                                  'author={Henry Kvinge and Elizabeth Coda and '
                                  'Eric Yeats and Davis Brown and John '
                                  'Buckheit and Sarah McGuire Scullen and '
                                  'Brendan Kennedy and Loc Truong and William '
                                  'Kay and Cliff Joslyn and Tegan Emerson and '
                                  'Michael J. Henry and John Anthony '
                                  '